<a href="https://colab.research.google.com/github/Sagnik-Chowdhury/Federated-Learning-1/blob/Sourit/Scaffolding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SCAFFOLD Optimization & Differential Privacy
## Advanced Drift Correction using Control Variates

**Objective:**
Standard FedAvg suffers from "client drift" when data is highly scattered. While FedProx addresses this via a proximal penalty, **SCAFFOLD** (Stochastic Controlled Averaging) corrects this drift directly within the gradient descent step using Control Variates.

**The Mechanism:**
1. The server maintains a global control variate ($c$) representing the overall direction of the global model.
2. Each client maintains a local control variate ($c_i$) representing their specific local data bias.
3. During local training, the client modifies its gradients: $g = g - c_i + c$. This physically steers the local optimizer away from its local bias and toward the global objective.
4. After training, the client updates its local control variate and sends the difference ($\Delta c_i$) back to the server alongside its weight updates.

**The Hypothesis:**
Because SCAFFOLD is so mathematically precise at correcting tabular variance, we expect the Breast Cancer dataset to remain near-perfect. However, the heavy-tailed Laplace noise injected by the server will still overwhelm the spatial pixel dependencies of the MNIST model, proving our core thesis that algorithmic stabilization cannot bypass fundamental data modality limits.

## Libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, TensorDataset
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
import copy
import pandas as pd

## Architecture

In [2]:
class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        return self.fc4(x)

In [3]:
class TabularNet(nn.Module):
    def __init__(self):
        super(TabularNet, self).__init__()
        self.fc1 = nn.Linear(30, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

## Data Preparation

In [4]:
NUM_CLIENTS = 20

In [5]:
print("Preparing MNIST Dataset (Integrated Data)")

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

mnist_full = datasets.MNIST('./data', train=True, download=True, transform=transform)
mnist_split = random_split(mnist_full, [len(mnist_full) // NUM_CLIENTS] * NUM_CLIENTS)
mnist_loaders = [DataLoader(ds, batch_size=32, shuffle=True) for ds in mnist_split]

mnist_test = datasets.MNIST('./data', train=False, download=True, transform=transform)
mnist_test_loader = DataLoader(mnist_test, batch_size=1000, shuffle=False)

print("\nData preparation complete")

Preparing MNIST Dataset (Integrated Data)


100%|██████████| 9.91M/9.91M [00:00<00:00, 41.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.26MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.4MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.99MB/s]



Data preparation complete


In [6]:
print("Preparing Breast Cancer Dataset (Scattered/Tabular Data)")

data = load_breast_cancer()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data.data)
y = data.target

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)
tabular_full = TensorDataset(X_tensor, y_tensor)

tab_split_size = len(tabular_full) // NUM_CLIENTS
tab_splits = [tab_split_size] * NUM_CLIENTS
tab_splits[-1] += len(tabular_full) % NUM_CLIENTS
tabular_loaders = [DataLoader(ds, batch_size=8, shuffle=True) for ds in random_split(tabular_full, tab_splits)]
tabular_test_loader = DataLoader(tabular_full, batch_size=len(tabular_full), shuffle=False)

print("\nData preparation complete")

Preparing Breast Cancer Dataset (Scattered/Tabular Data)

Data preparation complete


## The Defense Mechanism: SCAFFOLD Aggregation & Control Variates

Unlike standard aggregation methods that only average model weights, SCAFFOLD requires a dual-aggregation system.

**How it works:**
1. **Weight Aggregation:** The server averages the client weights exactly like standard FedAvg.

2. **Control Variate Aggregation:** The server collects the "drift deltas" ($\Delta c_i$) from every client. It averages these deltas and adds them to the **Global Control Variate** ($c$).

This global control variate essentially acts as a map of the true objective. In the next round, the server broadcasts this updated map back to the clients so they can correct their local gradients before they step off course.

In [11]:
def scaffold_aggregation(client_weights_list, control_deltas_list, global_control):
    """
    SCAFFOLD requires the server to aggregate TWO things:
    1. The standard model weights.
    2. The control variate deltas (to update the global drift map).
    """
    # 1. Aggregate Weights (Standard FedAvg)
    avg_weights = copy.deepcopy(client_weights_list[0])
    for key in avg_weights.keys():
        stacked_weights = torch.stack([client[key] for client in client_weights_list])
        avg_weights[key] = torch.mean(stacked_weights, dim=0)

    # 2. Aggregate Control Variates: c = c + (1/N) * sum(delta c_i)
    updated_global_control = copy.deepcopy(global_control)
    for name in updated_global_control.keys():
        stacked_deltas = torch.stack([delta[name] for delta in control_deltas_list])
        updated_global_control[name] += torch.mean(stacked_deltas, dim=0)

    return avg_weights, updated_global_control

In [12]:
def add_dp_noise(weights, noise_type='none', scale=0.01):
    """Injects Differential Privacy noise into the aggregated weights."""
    if noise_type == 'none':
        return weights
    noisy_weights = copy.deepcopy(weights)
    for key in noisy_weights.keys():
        tensor = noisy_weights[key]
        if noise_type == 'normal':
            noise = torch.randn_like(tensor) * scale
        elif noise_type == 'laplace':
            m = torch.distributions.laplace.Laplace(torch.tensor([0.0]), torch.tensor([scale]))
            noise = m.sample(tensor.shape).squeeze(-1).to(tensor.device)
        noisy_weights[key] = tensor + noise
    return noisy_weights

## Automated Grid Search Execution (SCAFFOLD)

This loop executes the SCAFFOLD training phase across our two data modalities.

During local training, each client utilizes its own **Local Control Variate** ($c_i$) alongside the global one ($c$) to physically alter the PyTorch gradients ($g = g - c_i + c$).

* **Federated Rounds:** 5
* **DP Noise Scale:** 0.05
* **Noise Types:** Baseline (None), Gaussian (Normal), Laplace

In [15]:
datasets_to_test = ['mnist', 'tabular']
noise_types_to_test = ['none', 'normal', 'laplace']
NOISE_SCALE = 0.05
federated_rounds = 5
epochs_per_round = 1

experiment_results = {'mnist': {}, 'tabular': {}}

print("Starting Automated Grid Search for SCAFFOLD")

for dataset in datasets_to_test:
    for noise in noise_types_to_test:
        print(f"\nTesting {dataset.upper()} with {noise.upper()} noise ")

        # 1. SETUP & RESET THE MODEL
        if dataset == 'mnist':
            global_model = MNISTNet()
            loaders = mnist_loaders
            test_loader = mnist_test_loader
            lr = 0.001
        else:
            global_model = TabularNet()
            loaders = tabular_loaders
            test_loader = tabular_test_loader
            lr = 0.01

        # --- SCAFFOLD INITIALIZATION ---
        global_control = {name: torch.zeros_like(param.data) for name, param in global_model.named_parameters()}
        local_controls = [{name: torch.zeros_like(param.data) for name, param in global_model.named_parameters()} for _ in range(NUM_CLIENTS)]

        # 2. THE EXECUTION LOOP
        for round_num in range(federated_rounds):
            client_weights = []
            control_deltas_list = []

            for client_idx in range(NUM_CLIENTS):
                local_model = MNISTNet() if dataset == 'mnist' else TabularNet()
                local_model.load_state_dict(global_model.state_dict())

                optimizer = optim.Adam(local_model.parameters(), lr=lr)
                criterion = nn.CrossEntropyLoss()
                local_model.train()

                num_steps = 0
                local_c = local_controls[client_idx]

                for epoch in range(epochs_per_round):
                    for inputs, labels in loaders[client_idx]:
                        optimizer.zero_grad()
                        outputs = local_model(inputs)
                        loss = criterion(outputs, labels)
                        loss.backward()

                        # SCAFFOLD CORE: Gradient Modification
                        for name, param in local_model.named_parameters():
                            param.grad.data += (global_control[name] - local_c[name])

                        optimizer.step()
                        num_steps += 1

                client_weights.append(local_model.state_dict())

                # SCAFFOLD CORE: Update Local Control Variate
                client_control_delta = {}
                global_weights = global_model.state_dict()

                for name, param in local_model.named_parameters():
                    weight_difference = global_weights[name] - param.data
                    new_c = local_c[name] - global_control[name] + (weight_difference / (num_steps * lr))
                    client_control_delta[name] = new_c - local_c[name]
                    local_controls[client_idx][name] = new_c

                control_deltas_list.append(client_control_delta)

            # --- USING OUR NEW SYSTEM FUNCTION ---
            aggregated_weights, global_control = scaffold_aggregation(
                client_weights,
                control_deltas_list,
                global_control
            )

            # Add DP Noise to the aggregated weights
            secured_weights = add_dp_noise(aggregated_weights, noise_type=noise, scale=NOISE_SCALE)
            global_model.load_state_dict(secured_weights)

        # 3. THE EVALUATION
        global_model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                outputs = global_model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        final_accuracy = 100 * correct / total
        print(f"Accuracy: {final_accuracy:.2f}% ")
        experiment_results[dataset][noise] = final_accuracy


Starting Automated Grid Search for SCAFFOLD

Testing MNIST with NONE noise 
Accuracy: 15.87% 

Testing MNIST with NORMAL noise 
Accuracy: 11.65% 

Testing MNIST with LAPLACE noise 
Accuracy: 9.22% 

Testing TABULAR with NONE noise 
Accuracy: 93.50% 

Testing TABULAR with NORMAL noise 
Accuracy: 88.22% 

Testing TABULAR with LAPLACE noise 
Accuracy: 81.90% 


In [16]:
print("\n" + "="*50)
print("SCAFFOLD RESULTS")
print("="*50)

results_df = pd.DataFrame(experiment_results).T
results_df.columns = ['Baseline (No DP)', 'Normal (Gaussian)', 'Laplace']
results_df.index = ['MNIST (Dense)', 'Breast Cancer (Scattered)']

print(results_df.to_string())
print("="*50)


SCAFFOLD RESULTS
                           Baseline (No DP)  Normal (Gaussian)    Laplace
MNIST (Dense)                     15.870000          11.650000   9.220000
Breast Cancer (Scattered)         93.497364          88.224956  81.898067


## Final Observations & Conclusion: SCAFFOLD Strategy

The integration of **SCAFFOLD** highlighted a critical reality of advanced federated optimization: algorithmic complexity does not guarantee immediate stability.

1. **The "Cold Start" Vulnerability:** Because we limited the simulation to 5 federated rounds, SCAFFOLD's control variates did not have enough time to accurately map and correct the client drift. These early, uncalibrated gradient corrections caused the complex MNIST model to completely collapse (dropping to a baseline of 15.87%). Applying DP Laplace noise further degraded it to 9.22% (random guessing).

2. **Scattered Data Remains Resilient:** Despite the unstable gradient corrections that destroyed the image model, the tabular Breast Cancer dataset demonstrated its incredible structural resilience. It maintained a highly respectable 93.49% baseline and comfortably absorbed the aggressive Laplace noise, holding strong at 81.89%.

**Grand Conclusion for Algolabs Project Phase 3:**
Across four distinct defense mechanisms (Trimmed Mean, Gradient Clipping, FedProx, and SCAFFOLD), the mathematical consensus is absolute. **Privacy architectures must be modality-aware.** Complex spatial data (Images) cannot survive heavy-tailed Differential Privacy or highly unstable gradient modifications. Conversely, scattered tabular features remain fundamentally robust, allowing for the deployment of advanced privacy mechanisms with minimal utility loss.